# Build an Adversarial Stop Sign Attack Pipeline

This notebook demonstrates an inference-time evasion workflow against a traffic sign classifier. The demo uses a downloadable Vision Transformer model fine-tuned on GTSRB, which includes a real `Stop` class.

We compare SimBA as a black-box attack with Projected Gradient Descent as a white-box attack, then review the tradeoff between perturbation strength, model confidence, and visual stealth.

## 1. Imports and Paths

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import datasets

DEMO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("module-7-apply-evasion-attacks/demo")
ART_HOME = DEMO_ROOT / "art_data"
ART_HOME.mkdir(parents=True, exist_ok=True)
os.environ["ART_DATA_PATH"] = str(ART_HOME)
os.environ["HOME"] = str(DEMO_ROOT)
os.environ["USERPROFILE"] = str(DEMO_ROOT)
sys.path.append(str(DEMO_ROOT / "src"))

from vision_attack_utils import (
    GTSRB_CLASSES,
    STOP_CLASS_ID,
    get_first_class_image,
    load_gtsrb_vit,
    perturbation_metrics,
    pil_to_raw_array,
    plot_attack_comparison,
    predict_numpy,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Load the Downloaded GTSRB Model

Before running this cell, download the model files from the demo root:

```bash
bash scripts/download_gtsrb_model.sh
```

The files should appear in `downloads/gtsrb_model/`.

In [ ]:
model_dir = DEMO_ROOT / "downloads/gtsrb_model"
model, processor = load_gtsrb_vit(model_dir, device=device)

print(f"Loaded GTSRB model from: {model_dir}")
print(f"Stop class id: {STOP_CLASS_ID} ({GTSRB_CLASSES[STOP_CLASS_ID]})")

## 3. Load GTSRB and Evaluate a Clean Baseline

This quick baseline checks model behavior on a small subset before adversarial testing. Clean accuracy is useful context, but it is not a robustness guarantee.

In [ ]:
test_set = datasets.GTSRB(
    root=str(DEMO_ROOT / "data"),
    split="test",
    download=True,
)

correct = 0
total = 0
for idx, (image, label) in enumerate(test_set):
    if idx >= 200:
        break
    x = pil_to_raw_array(image, size=224)
    pred_idx, _, _ = predict_numpy(model, x, device=device)
    correct += int(pred_idx[0] == label)
    total += 1

clean_accuracy = correct / total if total else 0.0
print(f"Clean subset accuracy: {clean_accuracy:.3f} over {total} GTSRB test images")

## 4. Select a Stop Sign Image

GTSRB class `14` is `Stop`. For a short classroom demo, we use a correctly classified stop sign with moderate confidence so both attacks can show a visible confidence change without a long search.

In [ ]:
stop_sample_index = 1277
stop_image_pil, stop_label = test_set[stop_sample_index]
assert stop_label == STOP_CLASS_ID, f"Expected GTSRB stop class {STOP_CLASS_ID}, got {stop_label}"
clean_image = pil_to_raw_array(stop_image_pil, size=224)

label_idx, confidence, _ = predict_numpy(model, clean_image, device=device)
clean_label = GTSRB_CLASSES[int(label_idx[0])]
clean_confidence = float(confidence[0])

plt.figure(figsize=(3, 3))
plt.imshow(np.transpose(clean_image[0], (1, 2, 0)))
plt.title(f"Clean GTSRB image\nPrediction: {clean_label} ({clean_confidence:.3f})")
plt.axis("off")
plt.show()

## 5. Run SimBA as a Black-Box Attack

SimBA uses model queries to reduce confidence in the original class. It does not require gradient access. The DCT attack mode perturbs frequency components, which is faster than searching individual pixels for this 224 x 224 image.

In [ ]:
from art.attacks.evasion import SimBA
from art.estimators.classification import PyTorchClassifier

class ProbabilityModel(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        return torch.softmax(self.base_model(x), dim=1)

probability_model = ProbabilityModel(model).to(device).eval()

criterion = torch.nn.CrossEntropyLoss()
classifier = PyTorchClassifier(
    model=probability_model,
    loss=criterion,
    input_shape=(3, 224, 224),
    nb_classes=len(GTSRB_CLASSES),
    clip_values=(0.0, 1.0),
)

simba = SimBA(
    classifier=classifier,
    attack="dct",
    max_iter=100,
    epsilon=0.10,
    batch_size=1,
)

simba_adv = simba.generate(x=clean_image)
simba_label_idx, simba_confidence, _ = predict_numpy(model, simba_adv, device=device)
simba_label = GTSRB_CLASSES[int(simba_label_idx[0])]
simba_metrics = perturbation_metrics(clean_image, simba_adv)

print(f"SimBA prediction: {simba_label} ({float(simba_confidence[0]):.3f})")
print(simba_metrics)

## 6. Implement PGD as a White-Box Attack

PGD uses gradients to search for an adversarial image inside an epsilon-bounded perturbation budget.

In [ ]:
def pgd_untargeted(model, x_np, y_np, epsilon=0.02, step_size=0.004, steps=20, device="cpu"):
    x_clean = torch.tensor(x_np, dtype=torch.float32, device=device)
    y = torch.tensor(y_np, dtype=torch.long, device=device)
    x_adv = x_clean.clone().detach()

    for _ in range(steps):
        x_adv.requires_grad_(True)
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x_adv)[0]

        with torch.no_grad():
            x_adv = x_adv + step_size * grad.sign()
            delta = torch.clamp(x_adv - x_clean, min=-epsilon, max=epsilon)
            x_adv = torch.clamp(x_clean + delta, min=0.0, max=1.0)

    return x_adv.detach().cpu().numpy()

pgd_adv = pgd_untargeted(
    model,
    clean_image,
    y_np=label_idx,
    epsilon=0.02,
    step_size=0.004,
    steps=20,
    device=device,
)

pgd_label_idx, pgd_confidence, _ = predict_numpy(model, pgd_adv, device=device)
pgd_label = GTSRB_CLASSES[int(pgd_label_idx[0])]
pgd_metrics = perturbation_metrics(clean_image, pgd_adv)

print(f"PGD prediction: {pgd_label} ({float(pgd_confidence[0]):.3f})")
print(pgd_metrics)

## 7. Compare Clean, SimBA, and PGD Results

In [ ]:
comparison_labels = [
    f"{clean_label}: {clean_confidence:.3f}",
    f"{simba_label}: {float(simba_confidence[0]):.3f}",
    f"{pgd_label}: {float(pgd_confidence[0]):.3f}",
]

plot_path = plot_attack_comparison(
    clean_image,
    simba_adv,
    pgd_adv,
    comparison_labels,
    DEMO_ROOT / "results/attack_comparison.png",
)

rows = [
    ["Clean", clean_label, clean_confidence, 0.0, 0.0],
    ["SimBA", simba_label, float(simba_confidence[0]), simba_metrics["linf"], simba_metrics["l2"]],
    ["PGD", pgd_label, float(pgd_confidence[0]), pgd_metrics["linf"], pgd_metrics["l2"]],
]

print("| Image | Prediction | Confidence | Linf Perturbation | L2 Perturbation |")
print("|-------|------------|------------|-------------------|-----------------|")
for name, pred, conf, linf, l2 in rows:
    print(f"| {name} | {pred} | {conf:.3f} | {linf:.4f} | {l2:.4f} |")

print(f"\nSaved comparison plot to: {plot_path}")

## 8. Exercise: Tune Epsilon and Discuss Operational Risk

Try changing the SimBA and PGD epsilon values. Larger epsilon values often improve attack success, but they also make perturbations more visible. In a warehouse robotics setting, the operational question is not only whether a model is accurate on clean images, but whether it remains reliable when camera inputs are slightly altered by noise, lighting, stickers, compression, or deliberate perturbations.